# Generated Python Notebook

This notebook was exported from the web app.

- Modify the code below
- Run cells as usual in Jupyter


In [ ]:
!pip install quantvn pandas numpy

In [ ]:
import numpy as np
import pandas as pd

def gen_position(df: pd.DataFrame) -> pd.DataFrame:

    # EMA FAST
    df['ema_12'] = df['Close'].ewm(
        span=12,
        adjust=False
    ).mean()

    # EMA SLOW
    df['ema_26'] = df['Close'].ewm(
        span=26,
        adjust=False
    ).mean()

    # EMA TREND
    df['ema_50'] = df['Close'].ewm(
        span=50,
        adjust=False
    ).mean()

    # MACD
    df['macd'] = (
        df['ema_12']
        -
        df['ema_26']
    )

    # SIGNAL LINE
    df['signal_line'] = df['macd'].ewm(
        span=9,
        adjust=False
    ).mean()

    # HISTOGRAM
    df['histogram'] = (
        df['macd']
        -
        df['signal_line']
    )

    # VOLUME FILTER
    if 'Volume' in df.columns:

        df['vol_ma20'] = (
            df['Volume']
            .rolling(20)
            .mean()
        )

        volume_confirm = (
            df['Volume']
            >
            df['vol_ma20']
        )

    else:

        volume_confirm = True

    # BUY SIGNAL
    buy_signal = (

        (df['macd'] > df['signal_line'])

        &

        (
            df['macd'].shift(1)
            <=
            df['signal_line'].shift(1)
        )

        &

        (
            df['Close']
            >
            df['ema_50']
        )

        &

        (
            df['histogram']
            >
            0
        )

        &

        volume_confirm

    )

    # SELL SIGNAL
    sell_signal = (

        (
            df['macd']
            <
            df['signal_line']
        )

        |

        (
            df['Close']
            <
            df['ema_50']
        )

    )

    # SIGNAL
    df['signal'] = 0

    df.loc[
        buy_signal,
        'signal'
    ] = 1

    df.loc[
        sell_signal,
        'signal'
    ] = -1

    # POSITION
    df['position'] = np.nan

    df.loc[
        df['signal'] == 1,
        'position'
    ] = 1

    df.loc[
        df['signal'] == -1,
        'position'
    ] = 0

    df['position'] = (
        df['position']
        .ffill()
        .fillna(0)
    )

    return df

In [ ]:
from quantvn import client

client(apikey="2hcZRZcdO0t0MbtFV8h2RI0vIwSQmtB0Di98u2i5OF2tm2tIrmMC1dnJAxW1fFT3MmtipRm6V3ManYfBuCm3cnZCJoWh6rWaQ3dC2s0-VeFNNolI1iQl62wxGWtaIcJm")


from quantvn.vn.data import get_stock_hist
from quantvn.vn.metrics import Backtest_Stock, Metrics

df = get_stock_hist("VNM", "1h")
df_pos = gen_position(df)
backtest = Backtest_Stock(df_pos, pnl_type="raw")
backtest.PNL().plot()



# Khởi tạo metrics
metrics = Metrics(backtest)

# Tính các metrics cơ bản
print("Average Return:", metrics.avg_return())
print("Average Win:", metrics.avg_win())
print("Average Loss:", metrics.avg_loss())
print("Win Rate:", metrics.win_rate())
print("Volatility:", metrics.volatility())

print("\n=== Risk/Reward Metrics ===")
print("Sharpe Ratio:", metrics.sharpe())
print("Sortino Ratio:", metrics.sortino())
print("Calmar Ratio:", metrics.calmar())
print("Max Drawdown:", metrics.max_drawdown())
print("Profit Factor:", metrics.profit_factor())
print("Risk of Ruin:", metrics.risk_of_ruin())
print("Value at Risk:", metrics.value_at_risk())
